# Kaggle Notebook: Standard Model Adversarial Attack Evaluation
This notebook is fully self-contained and runs on Kaggle (with GPU accelerated environment).
It trains a standard ResNet-18 classifier on CIFAR-10 and evaluates its robustness against:
1. Clean images
2. Dense FGSM Attack
3. Dense PGD Attack (10 iterations)
4. Sparse Top-K PGD Attack with Dynamic Masking (for various k-ratios: 0.1, 0.3, 0.5)



In [1]:
import os
import time
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchvision.models as tv_models
import numpy as np
import pandas as pd
from tqdm import tqdm

# Set seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")



Using device: cuda


In [2]:
# 1. Define CIFAR-adapted ResNet-18
def make_cifar_resnet18(num_classes=10):
    model = tv_models.resnet18(num_classes=num_classes)
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    return model



In [3]:
# 2. Define Attack Algorithms
def fgsm_attack(model, images, labels, eps=8/255):
    images = images.clone().detach().to(images.device)
    labels = labels.to(images.device)
    loss_fn = nn.CrossEntropyLoss()
    
    images.requires_grad = True
    outputs = model(images)
    loss = loss_fn(outputs, labels)
    grad = torch.autograd.grad(loss, images, retain_graph=False, create_graph=False)[0]
    
    adv_images = images + eps * grad.sign()
    adv_images = torch.clamp(adv_images, min=0, max=1).detach()
    return adv_images

def pgd_attack(model, images, labels, eps=8/255, alpha=2/255, iters=10):
    images = images.clone().detach().to(images.device)
    labels = labels.to(images.device)
    loss_fn = nn.CrossEntropyLoss()
    
    adv_images = images + torch.empty_like(images).uniform_(-eps, eps)
    adv_images = torch.clamp(adv_images, min=0, max=1).detach()
    
    for _ in range(iters):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        loss = loss_fn(outputs, labels)
        grad = torch.autograd.grad(loss, adv_images, retain_graph=False, create_graph=False)[0]
        
        adv_images = adv_images.detach() + alpha * grad.sign()
        delta = torch.clamp(adv_images - images, min=-eps, max=eps)
        adv_images = torch.clamp(images + delta, min=0, max=1).detach()
    return adv_images

def topk_pgd_attack(model, images, labels, eps=8/255, alpha=2/255, iters=10, k_ratio=0.1, dynamic=True):
    images = images.clone().detach().to(images.device)
    labels = labels.to(images.device)
    loss_fn = nn.CrossEntropyLoss()
    
    adv_images = images.clone().detach()
    mask = None
    
    for t in range(iters):
        adv_images.requires_grad = True
        outputs = model(adv_images)
        loss = loss_fn(outputs, labels)
        grad = torch.autograd.grad(loss, adv_images, retain_graph=False, create_graph=False)[0]
        
        with torch.no_grad():
            if dynamic or (t == 0):
                if k_ratio == 0:
                    mask = torch.zeros_like(grad)
                else:
                    score = grad.abs()
                    score_flatten = score.view(score.size(0), -1)
                    N = score_flatten.size(1)
                    
                    k_max = int(N * k_ratio)
                    if dynamic:
                        k_t = int(k_max * (1 - t / iters))
                    else:
                        k_t = k_max
                    if k_t < 1: k_t = 1
                    
                    topk_vals, _ = torch.topk(score_flatten, k_t, dim=1)
                    tau = topk_vals[:, -1].view(-1, 1, 1, 1)
                    mask = (score >= tau).float()
                
            update = alpha * grad.sign() * mask
            adv_images.data.copy_(images + torch.clamp(adv_images + update - images, min=-eps, max=eps))
            adv_images.data.clamp_(0, 1)
            
    return adv_images




In [4]:
# 3. Load CIFAR-10 Data
import shutil
import tarfile

def prepare_cifar10():
    target_dir = './data'
    extracted_target = os.path.join(target_dir, 'cifar-10-batches-py')
    tar_target = os.path.join(target_dir, 'cifar-10-python.tar.gz')
    
    if os.path.exists(extracted_target):
        print("CIFAR-10 is already prepared.")
        return
    
    os.makedirs(target_dir, exist_ok=True)
    candidates = [
        '/kaggle/input/datasets/truongnhatquangk18dn/aadata/cifar-10-python',
        '/kaggle/input/aadata/cifar-10-python',
        '/kaggle/input/aadata',
        '/kaggle/input/cifar-10-python'
    ]
    
    found = False
    for candidate in candidates:
        if not os.path.exists(candidate):
            continue
        
        # Check if contains extracted files directly
        if os.path.exists(os.path.join(candidate, 'batches.meta')):
            print(f"Linking extracted CIFAR-10 from {candidate}")
            os.symlink(candidate, extracted_target)
            found = True
            break
        elif os.path.exists(os.path.join(candidate, 'cifar-10-batches-py')):
            print(f"Linking extracted CIFAR-10 from {candidate}")
            os.symlink(os.path.join(candidate, 'cifar-10-batches-py'), extracted_target)
            found = True
            break
            
        # Check if tar.gz exists
        tar_path = os.path.join(candidate, 'cifar-10-python.tar.gz')
        if os.path.isfile(tar_path):
            print(f"Copying and extracting {tar_path}")
            shutil.copy(tar_path, tar_target)
            with tarfile.open(tar_target, 'r:gz') as tar:
                tar.extractall(path=target_dir)
            found = True
            break
        elif os.path.isfile(candidate) and candidate.endswith('.tar.gz'):
            print(f"Copying and extracting {candidate}")
            shutil.copy(candidate, tar_target)
            with tarfile.open(tar_target, 'r:gz') as tar:
                tar.extractall(path=target_dir)
            found = True
            break

    if not found:
        print("Warning: Could not find local CIFAR-10 dataset candidate. Will download from torchvision if internet is enabled.")

# Prepare the dataset
prepare_cifar10()

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])
test_transform = transforms.Compose([
    transforms.ToTensor(),
])

print("Loading CIFAR-10 dataset...")
train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform)
test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2)



Linking extracted CIFAR-10 from /kaggle/input/datasets/truongnhatquangk18dn/aadata/cifar-10-python
Loading CIFAR-10 dataset...


In [5]:
# 4. Load Pre-trained Standard Model via RobustBench
try:
    import robustbench
except ImportError:
    print("Installing robustbench from GitHub...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/RobustBench/robustbench.git", "--quiet"])

from robustbench.utils import load_model

print("Loading standard ResNet-18 model from RobustBench...")
model = load_model(
    model_name='Standard',
    dataset='cifar10',
    threat_model='Linf'
).to(device)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs with DataParallel!")
    model = nn.DataParallel(model)
model.eval()
print("Model loaded successfully!")




Installing robustbench from GitHub...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 100.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cudf-cu12 26.2.1 requires numba-cuda[cu12]<0.23.0,>=0.22.2, but you hav

Loading standard ResNet-18 model from RobustBench...


Downloading...
From (original): https://drive.google.com/uc?id=1t98aEuzeTL8P7Kpd5DIrCoCL21BNZUhC
From (redirected): https://drive.google.com/uc?id=1t98aEuzeTL8P7Kpd5DIrCoCL21BNZUhC&confirm=t&uuid=562a0dc3-52b4-4447-a234-17e648973617
To: /kaggle/working/models/cifar10/Linf/Standard.pt
100%|██████████| 292M/292M [00:01<00:00, 210MB/s] 


Using 2 GPUs with DataParallel!
Model loaded successfully!


In [6]:
# 5. Evaluate Clean Accuracy
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

clean_acc = 100 * correct / total
print(f"\nClean Test Accuracy of Standard ResNet-18: {clean_acc:.2f}%")




Clean Test Accuracy of Standard ResNet-18: 94.78%


In [7]:
# 6. Evaluate Robustness Against Attacks (on 1000 samples for speed)
eval_size = 1000
eval_images = []
eval_labels = []
samples_collected = 0
for img, lbl in test_loader:
    if samples_collected >= eval_size:
        break
    size_to_add = min(eval_size - samples_collected, img.size(0))
    eval_images.append(img[:size_to_add])
    eval_labels.append(lbl[:size_to_add])
    samples_collected += size_to_add

eval_images = torch.cat(eval_images, dim=0).to(device)
eval_labels = torch.cat(eval_labels, dim=0).to(device)

def evaluate_attack(model, attack_fn, images, labels, batch_size=128, **kwargs):
    model.eval()
    correct = 0
    total = images.size(0)
    for i in range(0, total, batch_size):
        img_batch = images[i:i+batch_size]
        lbl_batch = labels[i:i+batch_size]
        adv_batch = attack_fn(model, img_batch, lbl_batch, **kwargs)
        with torch.no_grad():
            outputs = model(adv_batch)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == lbl_batch).sum().item()
    return 100 * correct / total

print(f"\nRunning adversarial evaluations on {eval_size} test samples...")

# A. FGSM
acc_fgsm = evaluate_attack(model, fgsm_attack, eval_images, eval_labels, batch_size=128, eps=8/255)
print(f"FGSM Robust Accuracy (eps=8/255): {acc_fgsm:.2f}%")

# B. PGD-10
acc_pgd = evaluate_attack(model, pgd_attack, eval_images, eval_labels, batch_size=128, eps=8/255, alpha=2/255, iters=10)
print(f"PGD-10 Robust Accuracy (eps=8/255, alpha=2/255): {acc_pgd:.2f}%")

# C. Sparse PGD (k = 0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0)
sparse_results = {}
k_values = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
for k in k_values:
    acc_sparse = evaluate_attack(model, topk_pgd_attack, eval_images, eval_labels, batch_size=128, eps=8/255, alpha=2/255, iters=10, k_ratio=k, dynamic=True)
    sparse_results[k] = acc_sparse
    print(f"Sparse PGD Robust Accuracy (k={k}): {acc_sparse:.2f}%")




Running adversarial evaluations on 1000 test samples...
FGSM Robust Accuracy (eps=8/255): 30.50%
PGD-10 Robust Accuracy (eps=8/255, alpha=2/255): 0.00%
Sparse PGD Robust Accuracy (k=0.0): 94.80%
Sparse PGD Robust Accuracy (k=0.1): 8.00%
Sparse PGD Robust Accuracy (k=0.2): 1.90%
Sparse PGD Robust Accuracy (k=0.3): 0.60%
Sparse PGD Robust Accuracy (k=0.4): 0.30%
Sparse PGD Robust Accuracy (k=0.5): 0.20%
Sparse PGD Robust Accuracy (k=0.6): 0.10%
Sparse PGD Robust Accuracy (k=0.7): 0.00%
Sparse PGD Robust Accuracy (k=0.8): 0.00%
Sparse PGD Robust Accuracy (k=0.9): 0.00%
Sparse PGD Robust Accuracy (k=1.0): 0.00%


In [8]:
# 7. Print Summary Report
results_summary = [
    {"Attack": "Clean (No Attack)", "Robust Accuracy": f"{clean_acc:.2f}%", "ASR": "0.00%"},
    {"Attack": "FGSM (eps=8/255)", "Robust Accuracy": f"{acc_fgsm:.2f}%", "ASR": f"{clean_acc - acc_fgsm:.2f}%"},
    {"Attack": "PGD-10 (eps=8/255)", "Robust Accuracy": f"{acc_pgd:.2f}%", "ASR": f"{clean_acc - acc_pgd:.2f}%"},
]
for k in k_values:
    results_summary.append({
        "Attack": f"Sparse PGD (k={k})",
        "Robust Accuracy": f"{sparse_results[k]:.2f}%",
        "ASR": f"{clean_acc - sparse_results[k]:.2f}%"
    })
df = pd.DataFrame(results_summary)
print("\n" + "="*50)
print(" STANDARD MODEL ADVERSARIAL ROBUSTNESS REPORT")
print("="*50)
print(df.to_string(index=False))
print("="*50)




 STANDARD MODEL ADVERSARIAL ROBUSTNESS REPORT
            Attack Robust Accuracy    ASR
 Clean (No Attack)          94.78%  0.00%
  FGSM (eps=8/255)          30.50% 64.28%
PGD-10 (eps=8/255)           0.00% 94.78%
Sparse PGD (k=0.0)          94.80% -0.02%
Sparse PGD (k=0.1)           8.00% 86.78%
Sparse PGD (k=0.2)           1.90% 92.88%
Sparse PGD (k=0.3)           0.60% 94.18%
Sparse PGD (k=0.4)           0.30% 94.48%
Sparse PGD (k=0.5)           0.20% 94.58%
Sparse PGD (k=0.6)           0.10% 94.68%
Sparse PGD (k=0.7)           0.00% 94.78%
Sparse PGD (k=0.8)           0.00% 94.78%
Sparse PGD (k=0.9)           0.00% 94.78%
Sparse PGD (k=1.0)           0.00% 94.78%
